# 03 - Xây dựng Fact Table tổng hợp



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from churn_prediction.paths import RAW_DIR

In [2]:
customer = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
geo = pd.read_csv(RAW_DIR / 'olist_geolocation_dataset.csv')
order = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv')
order_item = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
order_payment = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')
order_review = pd.read_csv(RAW_DIR / 'olist_order_reviews_dataset.csv')
product = pd.read_csv(RAW_DIR / 'olist_products_dataset.csv')
seller = pd.read_csv(RAW_DIR / 'olist_sellers_dataset.csv')
translation = pd.read_csv(RAW_DIR / 'product_category_name_translation.csv')


In [ ]:
# 2. Tiền xử lý các bảng có nhiều dòng: reviews và payments
# --- Reviews: lấy review_score trung bình (hoặc mới nhất) theo order_id
# Giả sử mỗi order_id có thể có nhiều review (do chỉnh sửa), lấy review mới nhất
order_review['review_creation_date'] = pd.to_datetime(order_review['review_creation_date'])
order_review_sorted = order_review.sort_values('review_creation_date')
order_review_latest = order_review_sorted.drop_duplicates('order_id', keep='last')
# Chỉ giữ các cột cần thiết
order_review_agg = order_review_latest[['order_id', 'review_score', 'review_comment_title', 'review_comment_message']]

# --- Payments: tổng số tiền, tổng số lần trả góp, phương thức thanh toán phổ biến
order_payment_agg = order_payment.groupby('order_id').agg(
    total_paid=('payment_value', 'sum'),
    total_installments=('payment_installments', 'sum'),
    payment_type=('payment_type', lambda x: x.mode()[0] if not x.empty else None)
).reset_index()

print(f"Reviews: {len(order_review_agg)} order_id unique")
print(f"Payments: {len(order_payment_agg)} order_id unique")

In [ ]:
# 3. Chuyển đổi kiểu dữ liệu cho các cột ngày trong orders
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    order[col] = pd.to_datetime(order[col], errors='coerce')

In [ ]:
# 4. Xây dựng fact table bằng cách merge tuần tự

# Bắt đầu từ order_items (mức chi tiết sản phẩm trong đơn)
fact = order_item.copy()

# Thêm sản phẩm
fact = fact.merge(product, on='product_id', how='left')
# Thêm dịch tên danh mục
fact = fact.merge(translation, on='product_category_name', how='left')
# Thêm người bán
fact = fact.merge(seller, on='seller_id', how='left')
# Thêm thông tin đơn hàng
fact = fact.merge(order, on='order_id', how='left')
# Thêm khách hàng
fact = fact.merge(customer, on='customer_id', how='left')
# Thêm thông tin đánh giá đã tổng hợp
fact = fact.merge(order_review_agg, on='order_id', how='left')
# Thêm thông tin thanh toán đã tổng hợp
fact = fact.merge(order_payment_agg, on='order_id', how='left')

print(f"Fact table shape: {fact.shape}")
print(f"Số lượng missing trong các cột chính:\n{fact[['price', 'freight_value', 'review_score', 'total_paid']].isnull().sum()}")

In [ ]:
# 5. Tính thêm một số feature hữu ích
# Thời gian giao hàng (nếu có)
fact['delivery_days'] = (fact['order_delivered_customer_date'] - fact['order_purchase_timestamp']).dt.days
# Độ trễ so với dự kiến
fact['delay_days'] = (fact['order_delivered_customer_date'] - fact['order_estimated_delivery_date']).dt.days
# Giá trị mỗi item (price + freight)
fact['item_total'] = fact['price'] + fact['freight_value']

print("Feature engineering hoàn tất.")

In [ ]:
# 6. Lưu fact table (đã clean và mở rộng) dưới dạng Parquet để phân tích sau
output_file = PROCESSED_PATH / 'fact_table.parquet'
fact.to_parquet(output_file, index=False)
print(f"Đã lưu fact table tại: {output_file}")
print(f"Dung lượng file: {output_file.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# 7. Kiểm tra nhanh một vài dòng để đảm bảo mọi thứ OK
fact.head(2).T

## Kết thúc
Bây giờ bạn có thể đọc file `fact_table.parquet` trong các notebook phân tích khác (churn, doanh thu, v.v.) mà không cần merge lại.